In [1]:
import json
import os
import re
import math
import random
import unicodedata
import zipfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------
# 0) IO
# ----------------------------
def read_lines(path):
    with open(path, "rt", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip("\n")
            if line.strip():
                yield line


# ----------------------------
# 1) Preprocess
# ----------------------------
ZERO_WIDTH_EXPLICIT = {
    "\u200b", "\u200c", "\u200d", "\ufeff", "\u2060",
}

def remove_zero_width(text):
    text = "".join(ch for ch in text if ch not in ZERO_WIDTH_EXPLICIT)
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Cf")
    return text

def is_punctuation(ch):
    return unicodedata.category(ch).startswith("P")

def separate_punctuation(text):
    out = []
    for ch in text:
        if is_punctuation(ch):
            out.append(" ")
            out.append(ch)
            out.append(" ")
        else:
            out.append(ch)
    return re.sub(r"\s+", " ", "".join(out)).strip()

def tokenize(text):
    text = unicodedata.normalize("NFKC", text)
    text = remove_zero_width(text)
    text = separate_punctuation(text)
    return text.split() if text else []


# ----------------------------
# 2) N-grams
# ----------------------------
def word_to_ngrams(word, min_n=3, max_n=5, bow="<", eow=">"):
    w = f"{bow}{word}{eow}"
    grams = [w]  # include special <word>
    L = len(w)
    for n in range(min_n, max_n + 1):
        if L < n:
            continue
        for i in range(0, L - n + 1):
            grams.append(w[i:i+n])
    return grams


# ----------------------------
# 3) Part 1: build ngram vocab + bags
# ----------------------------
def build_ngram_vocab(input_path, min_n=3, max_n=5, bow="<", eow=">", max_lines=None):
    ngrams_set = set()
    for li, line in enumerate(read_lines(input_path)):
        if max_lines is not None and li >= max_lines:
            break
        toks = tokenize(line)
        for tok in toks:
            grams = word_to_ngrams(tok, min_n, max_n, bow, eow)
            for g in grams:
                ngrams_set.add(g)

    id2ngram = sorted(list(ngrams_set))
    ngram2id = {}
    for i, g in enumerate(id2ngram):
        ngram2id[g] = i
    return ngram2id, id2ngram

def save_vocab_txt(lines, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wt", encoding="utf-8") as f:
        for x in lines:
            f.write(str(x) + "\n")

def save_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wt", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def write_bags_file(input_path, out_path, ngram2id, min_n=3, max_n=5, bow="<", eow=">", max_lines=None):
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    with open(out_path, "wt", encoding="utf-8") as out:
        for li, line in enumerate(read_lines(input_path)):
            if max_lines is not None and li >= max_lines:
                break
            toks = tokenize(line)
            bags = []
            for tok in toks:
                grams = word_to_ngrams(tok, min_n, max_n, bow, eow)
                ids = set()
                for g in grams:
                    if g in ngram2id:
                        ids.add(ngram2id[g])
                ids = sorted(list(ids))
                bags.append(",".join(map(str, ids)))
            out.write(" ".join(bags) + "\n")


# ----------------------------
# 3.5) Save vectors as TEXT
# ----------------------------
def save_vectors_txt(path, tokens, vecs, add_header=False):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wt", encoding="utf-8") as f:
        if add_header:
            f.write(f"{len(tokens)} {vecs.shape[1]}\n")
        for tok, v in zip(tokens, vecs):
            f.write(tok + " " + " ".join(f"{x:.6f}" for x in v.tolist()) + "\n")


# ----------------------------
# 4) Trainer helpers
# ----------------------------
def build_word_vocab(input_path, min_count=1, max_lines=None):
    from collections import Counter
    cnt = Counter()
    for li, line in enumerate(read_lines(input_path)):
        if max_lines is not None and li >= max_lines:
            break
        cnt.update(tokenize(line))

    words = sorted([w for w, c in cnt.items() if c >= min_count])
    word2id = {}
    for i, w in enumerate(words):
        word2id[w] = i
    counts = np.array([cnt[w] for w in words], dtype=np.int64)
    total = int(counts.sum())
    return word2id, words, counts, total

def compute_subsample_keep_prob(count, total, t):
    f = count / max(1, total)
    if f <= 0:
        return 1.0
    p = (math.sqrt(f / t) + 1.0) * (t / f)
    return float(min(1.0, p))

def negative_sampling_probs(counts, power=0.75):
    p = counts.astype(np.float64) ** power
    p /= (p.sum() + 1e-12)
    return p

def generate_skipgram_pairs(sentence_word_ids, window):
    pairs = []
    n = len(sentence_word_ids)
    for i, c in enumerate(sentence_word_ids):
        left = max(0, i - window)
        right = min(n, i + window + 1)
        for j in range(left, right):
            if j == i:
                continue
            pairs.append((c, sentence_word_ids[j]))
    return pairs


# ----------------------------
# 4) Model
# ----------------------------
class FastTextSGNS(nn.Module):
    def __init__(self, num_ngrams, num_words, dim):
        super().__init__()
        self.ngram_in = nn.Embedding(num_ngrams, dim)
        self.word_out = nn.Embedding(num_words, dim)
        nn.init.uniform_(self.ngram_in.weight, -0.5 / dim, 0.5 / dim)
        nn.init.zeros_(self.word_out.weight)

    def center_vec(self, ngram_ids_1d):
        return self.ngram_in(ngram_ids_1d).sum(dim=0)

    def sgns_loss(self, center_ngram_tensors, pos_ctx, neg_ctx):
        vc = torch.stack([self.center_vec(t) for t in center_ngram_tensors], dim=0)  # (B,D)

        u_pos = self.word_out(pos_ctx)                      # (B,D)
        pos_score = (vc * u_pos).sum(dim=1)                 # (B,)
        pos_term = F.logsigmoid(pos_score)

        u_neg = self.word_out(neg_ctx)                      # (B,K,D)
        neg_score = (u_neg * vc.unsqueeze(1)).sum(dim=2)    # (B,K)
        neg_term = F.logsigmoid(-neg_score).sum(dim=1)

        return -(pos_term + neg_term).mean()


# ----------------------------
# 4) Train
# ----------------------------
def train_fasttext(input_path, workdir, ngram2id, id2ngram,
                   dim=100, window=3, neg_k=3, epochs=2, lr=0.09,
                   min_count=1, subsample_t=1e-5, batch_size=1024,
                   max_lines=None, seed=13, device=None):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    os.makedirs(workdir, exist_ok=True)

    # word vocab (still needed)
    word2id, id2word, counts, total = build_word_vocab(input_path, min_count=min_count, max_lines=max_lines)

    save_vocab_txt(id2word, os.path.join(workdir, "word_vocab.txt"))
    save_json(word2id, os.path.join(workdir, "word2id.json"))
    save_vocab_txt(id2ngram, os.path.join(workdir, "ngram_vocab.txt"))
    save_json(ngram2id, os.path.join(workdir, "ngram2id.json"))

    # precompute word -> ngram tensor (ON GPU)
    word_ngram_tensors = []
    for w in id2word:
        grams = set(word_to_ngrams(w, 3, 5, "<", ">"))
        ids = []
        for g in grams:
            if g in ngram2id:
                ids.append(ngram2id[g])
        ids = sorted(ids)
        if len(ids) == 0:
            ids = [0]
        word_ngram_tensors.append(torch.tensor(ids, dtype=torch.long, device=device))

    # subsampling
    keep_prob = np.ones(len(id2word), dtype=np.float64)
    for i, c in enumerate(counts):
        keep_prob[i] = compute_subsample_keep_prob(int(c), total, subsample_t)

    # negative sampling dist (GPU)
    neg_p = negative_sampling_probs(counts, power=0.75)
    neg_p_t = torch.tensor(neg_p, dtype=torch.float32, device=device)

    model = FastTextSGNS(num_ngrams=len(id2ngram), num_words=len(id2word), dim=dim).to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr)

    model.train()
    step = 0

    for ep in range(epochs):
        for li, line in enumerate(read_lines(input_path)):
            if max_lines is not None and li >= max_lines:
                break

            toks = tokenize(line)

            sent_ids = []
            for w in toks:
                if w in word2id:
                    wid = word2id[w]
                    if random.random() <= keep_prob[wid]:
                        sent_ids.append(wid)

            if len(sent_ids) < 2:
                continue

            pairs = generate_skipgram_pairs(sent_ids, window=window)
            if len(pairs) == 0:
                continue

            for b0 in range(0, len(pairs), batch_size):
                batch = pairs[b0:b0 + batch_size]
                if len(batch) == 0:
                    continue

                centers = [c for (c, _) in batch]
                pos_ctx = torch.tensor([p for (_, p) in batch], dtype=torch.long, device=device)

                center_ngs = [word_ngram_tensors[c] for c in centers]

                B = len(batch)
                neg_flat = torch.multinomial(neg_p_t, num_samples=B * neg_k, replacement=True)
                neg_ctx = neg_flat.view(B, neg_k)

                opt.zero_grad(set_to_none=True)
                loss = model.sgns_loss(center_ngs, pos_ctx, neg_ctx)
                loss.backward()
                opt.step()

                step += 1
                if step % 200 == 0:
                    print(f"epoch={ep+1}/{epochs} line={li} step={step} loss={loss.item():.4f}")

    # save vectors
    ngram_vecs = model.ngram_in.weight.detach().cpu().numpy()
    out_word_vecs = model.word_out.weight.detach().cpu().numpy()

    word_vecs = np.zeros((len(id2word), dim), dtype=np.float32)
    for i in range(len(id2word)):
        ids_cpu = word_ngram_tensors[i].detach().cpu().numpy()
        word_vecs[i] = ngram_vecs[ids_cpu].sum(axis=0)

    save_vectors_txt(os.path.join(workdir, "ngram_vectors.txt"), id2ngram, ngram_vecs, add_header=False)
    save_vectors_txt(os.path.join(workdir, "out_word_vectors.txt"), id2word, out_word_vecs, add_header=False)
    save_vectors_txt(os.path.join(workdir, "word_vectors.txt"), id2word, word_vecs, add_header=False)

    ckpt_path = os.path.join(workdir, "fasttext_sgns_state_dict.pt")
    torch.save(model.state_dict(), ckpt_path)
    print("Saved model weights to:", ckpt_path)

    print("Training complete. Saved artifacts to:", workdir)
    print("Device used:", device)


# ----------------------------
# Zip helper
# ----------------------------
def zip_output_dir(output_dir, zip_path):
    output_dir = os.path.abspath(output_dir)
    zip_path = os.path.abspath(zip_path)

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(output_dir):
            for name in files:
                full_path = os.path.join(root, name)
                arcname = os.path.relpath(full_path, os.path.dirname(output_dir))
                zf.write(full_path, arcname)

    return zip_path


# ----------------------------
# MAIN
# ----------------------------
def main():
    input_file_path = "/kaggle/input/training-corpus/wmt-news-crawl-hi.txt"
    workdir = "/kaggle/working/q2_out"
    out_bags = os.path.join(workdir, "bags.txt")

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Part 1
    ngram2id, id2ngram = build_ngram_vocab(input_file_path, min_n=3, max_n=5, bow="<", eow=">")
    save_vocab_txt(id2ngram, os.path.join(workdir, "ngram_vocab.txt"))
    save_json(ngram2id, os.path.join(workdir, "ngram2id.json"))
    write_bags_file(input_file_path, out_bags, ngram2id, min_n=3, max_n=5, bow="<", eow=">")

    print("Part 1 done!")
    print("Saved ngram vocab to:", os.path.join(workdir, "ngram_vocab.txt"))
    print("Saved bags to      :", out_bags)

    # Train
    train_fasttext(
        input_path=input_file_path,
        workdir=workdir,
        ngram2id=ngram2id,
        id2ngram=id2ngram,
        dim=120,
        window=5,
        neg_k=3,
        epochs=3,
        lr=0.09,
        min_count=1,
        subsample_t=1e-5,
        batch_size=1024,
        device=device,
    )

    zip_output_dir(workdir, "/kaggle/working/q2_out.zip")
    print("Zipped outputs to:", "/kaggle/working/q2_out.zip")


if __name__ == "__main__":
    main()


Part 1 done!
Saved ngram vocab to: /kaggle/working/q2_out/ngram_vocab.txt
Saved bags to      : /kaggle/working/q2_out/bags.txt
epoch=1/3 line=223 step=200 loss=2.7726
epoch=1/3 line=447 step=400 loss=2.7726
epoch=1/3 line=675 step=600 loss=2.7726
epoch=1/3 line=897 step=800 loss=2.7726
epoch=1/3 line=1127 step=1000 loss=2.7726
epoch=1/3 line=1356 step=1200 loss=2.7725
epoch=1/3 line=1590 step=1400 loss=2.7726
epoch=1/3 line=1832 step=1600 loss=2.7726
epoch=1/3 line=2057 step=1800 loss=2.7726
epoch=1/3 line=2291 step=2000 loss=2.7726
epoch=1/3 line=2519 step=2200 loss=2.7725
epoch=1/3 line=2747 step=2400 loss=2.7726
epoch=1/3 line=2966 step=2600 loss=2.7726
epoch=1/3 line=3196 step=2800 loss=2.7725
epoch=1/3 line=3423 step=3000 loss=2.7725
epoch=1/3 line=3657 step=3200 loss=2.7726
epoch=1/3 line=3885 step=3400 loss=2.7724
epoch=1/3 line=4114 step=3600 loss=2.7725
epoch=1/3 line=4344 step=3800 loss=2.7725
epoch=1/3 line=4570 step=4000 loss=2.7725
epoch=1/3 line=4796 step=4200 loss=2.7721

In [1]:
#Experiment 1 => FastVec can give embeddings for OOV words also =>
import numpy as np
def word_to_ngrams(word, min_n=3, max_n=5, bow="<", eow=">"):
    w = f"{bow}{word}{eow}"
    grams = [w]  # include special <word>
    L = len(w)
    for n in range(min_n, max_n + 1):
        if L < n:
            continue
        for i in range(0, L - n + 1):
            grams.append(w[i:i+n])
    return grams


real_word = "अंतरिक्ष"

vocabulary_path = "/kaggle/input/outputs/word_vocab.txt"
word_vectors = "/kaggle/input/outputs/word_vectors.txt"
ngram_vectors = "/kaggle/input/outputs/ngram_vectors.txt"

# ---------- load vocab ----------
vocabulary = open(vocabulary_path, "r", encoding="utf-8")
words = [w.strip() for w in vocabulary.readlines()]   # strip newline
vocabulary.close()

# ---------- if word exists in vocab, fetch its learned vector ----------
if real_word in words:
    print("The word is in the vocabulary, Hence it will have a learned vector. The vector is as follows : ")

    word_embeddings = open(word_vectors, "r", encoding="utf-8")
    embeddings = word_embeddings.readlines()
    word_embeddings.close()

    for line in embeddings:
        parts = line.split()
        if not parts:
            continue
        if parts[0] == real_word:  # compare full token, not first char
            print(np.array(list(map(float, parts[1:])), dtype=np.float32))
            break

else:
    print("The word is not in the vocabulary, Hence it will not have a learned vector.")
    print("However we can split the word into its ngrams, and then use ngrams already present to create a representation:")

    ngrams_fake_word = word_to_ngrams(real_word)  # your function

    data = open(ngram_vectors, "r", encoding="utf-8")
    ngrams_embeddings = data.readlines()
    data.close()

    # Build lookup: ngram -> vector
    ngram2vec = {}
    for line in ngrams_embeddings:
        parts = line.split()
        if not parts:
            continue
        ngram2vec[parts[0]] = np.array(list(map(float, parts[1:])), dtype=np.float32)

    # Collect vectors for ngrams that exist
    ngrams_obtained = [ngram2vec[g] for g in ngrams_fake_word if g in ngram2vec]

    if len(ngrams_obtained) == 0:
        print("None of the n-grams were found in ngram_vectors.txt, cannot build representation.")
    else:
        final_vector_representation = np.sum(ngrams_obtained, axis=0)
        print(final_vector_representation)

The word is in the vocabulary, Hence it will have a learned vector. The vector is as follows : 
[-5.47900e-03 -3.17664e-01  5.37150e-02  2.08825e-01  2.97743e-01
  4.65769e-01 -1.62509e-01  5.45310e-02  1.36559e-01  1.39910e-02
 -1.78676e-01  2.24910e-02 -9.91000e-02  1.41361e-01 -1.69454e-01
 -4.38880e-02 -8.66070e-02 -2.87270e-02  1.61850e-01 -4.16152e-01
  2.23206e-01  2.25859e-01  1.60428e-01  5.81370e-02  8.65060e-02
 -7.63580e-02 -8.74000e-04  3.64576e-01 -2.82637e-01  1.97337e-01
  1.15767e-01  2.37694e-01  1.74800e-02 -2.17280e-02  6.11410e-02
 -2.01795e-01  3.23978e-01 -8.99200e-03  1.59022e-01 -1.82409e-01
  2.20717e-01 -2.24550e-02 -3.61717e-01  3.94831e-01  2.84234e-01
  1.13874e-01 -7.51600e-03 -1.81569e-01  4.76800e-02  1.46610e-02
 -2.09112e-01 -1.33242e-01  3.40299e-01 -2.74383e-01 -8.23797e-01
  5.30000e-05  8.38150e-02  5.03549e-01  2.03911e-01 -1.52533e-01
 -1.39476e-01  1.24140e-01  1.26223e-01 -1.75003e-01 -4.58160e-02
  2.53011e-01 -7.21730e-02  4.29810e-02  3.045

In [2]:
#Morphemes Test
def cosine_similarity(a: np.ndarray, b: np.ndarray, eps: float = 1e-12) -> float:
    a = np.asarray(a, dtype=np.float64).ravel()
    b = np.asarray(b, dtype=np.float64).ravel()

    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na < eps or nb < eps:
        return 0.0

    return float(np.dot(a, b) / (na * nb))

random_word = 'अंतरिक्ष' # strip newline if present
ngram_of_the_word = word_to_ngrams(random_word)

word_embeddings = open(word_vectors, "r", encoding="utf-8")
embeddings = word_embeddings.readlines()
word_embeddings.close()

random_word_embedding = None
for line in embeddings:
    parts = line.split()
    if not parts:
        continue
    if parts[0] == random_word:  # FIX: compare with random_word
        random_word_embedding = np.array(list(map(float, parts[1:])), dtype=np.float32)
        break

if random_word_embedding is None:
    raise ValueError(f"Could not find embedding for word: {random_word}")

data = open(ngram_vectors, "r", encoding="utf-8")
ngrams_embeddings = data.readlines()
data.close()

ngram2vec = {}
for line in ngrams_embeddings:
    parts = line.split()
    if not parts:
        continue
    ngram2vec[parts[0]] = np.array(list(map(float, parts[1:])), dtype=np.float32)

ngram_embedding_score = {}
for i in ngram_of_the_word:
    if i not in ngram2vec:   
        continue
    x = ngram2vec[i]
    resulting_embedding = random_word_embedding - x
    cosine_similarity_resultant = cosine_similarity(resulting_embedding, random_word_embedding)
    ngram_embedding_score.update({i: cosine_similarity_resultant})

sorted_items_desc = sorted(ngram_embedding_score.items(), key=lambda kv: kv[1], reverse=False)

print(random_word)
print(sorted_items_desc)

अंतरिक्ष
[('क्ष', 0.9997913239387543), ('<अं', 0.9998878782892012), ('िक्', 0.9999258074993799), ('ंतर', 0.9999370328027962), ('क्ष>', 0.99994069960547), ('रिक्', 0.9999420652116303), ('िक्ष', 0.9999439628887072), ('तरिक्', 0.999944722543707), ('तरि', 0.9999447611082778), ('<अंतरिक्ष>', 0.9999470683187165), ('अंत', 0.9999477779749275), ('<अंत', 0.9999483151245109), ('रिक', 0.9999483771899396), ('<अंतर', 0.99994854559436), ('अंतरि', 0.999950149254648), ('रिक्ष', 0.9999502964154813), ('अंतर', 0.9999508863317352), ('ंतरिक', 0.9999511081483942), ('्ष>', 0.9999514705540927), ('तरिक', 0.999954058821379), ('ंतरि', 0.9999555150856028), ('िक्ष>', 0.9999556764872126)]
